In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import imageio_ffmpeg
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

# Wavefunction Tunneling 

In [ ]:
x_min, x_max = -200, 200  
dx = 0.8                  
dt = 0.5                  

b_center = 0              
b_height = 0.6            
b_width = 5              


x0 = -150                 
sig0 = 5                  
k0 = 1                   


hbar = 1
m = 1


x = np.arange(x_min, x_max, dx)
N = len(x)

In [ ]:
def V_pot(x, center, height, width):
    V = np.zeros_like(x)
    mask = (x >= center - width/2) & (x <= center + width/2)
    V[mask] = height
    return V

V = V_pot(x, b_center, b_height, b_width)


def psi_init(x, x0, sig0, k0):
    norm = (2 * np.pi * sig0**2)**(-1/4)
    psi = norm * np.exp(-(x - x0)**2 / (4 * sig0**2)) * np.exp(1j * k0 * x)
    return psi

psi0 = psi_init(x, x0, sig0, k0)


def build_H(N, dx, V, hbar=1, m=1):
    
    T = np.zeros((N, N), dtype=complex)
    coeff = -hbar**2 / (2 * m * dx**2)
    
    for i in range(1, N-1):
        T[i, i-1] = coeff
        T[i, i] = -2 * coeff
        T[i, i+1] = coeff
    
  
    T[0, 0] = -2 * coeff
    T[0, 1] = coeff
    T[N-1, N-2] = coeff
    T[N-1, N-1] = -2 * coeff
    
    
    V_mat = np.diag(V)
    
    H = T + V_mat
    return H


In [ ]:
def time_evol_op(H, dt, hbar=1):
    I = np.eye(H.shape[0], dtype=complex)
    U = np.linalg.inv(I + 1j * dt/(2*hbar) * H) @ (I - 1j * dt/(2*hbar) * H)
    return U


H = build_H(N, dx, V, hbar, m)
U = time_evol_op(H, dt, hbar)


def sim_wave(psi0, U, nt=1000):
    psi_hist = []
    psi_curr = psi0.copy()
    
    for n in range(nt):
        psi_hist.append(psi_curr.copy())
    
        psi_next = U @ psi_curr
        psi_curr = psi_next
    
    return psi_hist

In [ ]:
print("Simulating quantum tunneling...")
psi_hist = sim_wave(psi0, U, nt=800)


def create_tun_anim(psi_hist, x, V, case_name, frame_step=10):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    prob_den = np.abs(psi_hist[0])**2
    line1, = ax1.plot(x, prob_den, 'b-', linewidth=2, label='|ψ|²')
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(0, max(np.max(np.abs(psi)**2) for psi in psi_hist) * 1.1)
    ax1.set_ylabel('Probability Density')
    ax1.set_title(f'Quantum Tunneling - {case_name}')
    ax1.grid(True)
    ax1.legend()
    
    ax1_twin = ax1.twinx()
    ax1_twin.plot(x, V, 'r-', linewidth=2, alpha=0.7, label='Potential')
    ax1_twin.set_ylabel('Potential Energy')
    ax1_twin.set_ylim(0, b_height * 1.2)
    ax1_twin.legend(loc='upper right')
    

    real_plt, = ax2.plot(x, psi_hist[0].real, 'g-', linewidth=1, label='Re(ψ)', alpha=0.7)
    imag_plt, = ax2.plot(x, psi_hist[0].imag, 'm-', linewidth=1, label='Im(ψ)', alpha=0.7)
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(-max(np.max(np.abs(psi)) for psi in psi_hist) * 1.1, 
                 max(np.max(np.abs(psi)) for psi in psi_hist) * 1.1)
    ax2.set_xlabel('Position x')
    ax2.set_ylabel('Wavefunction')
    ax2.grid(True)
    ax2.legend()
    

    frames = range(0, len(psi_hist), frame_step)
    
    def animate(i):
      
        prob_den = np.abs(psi_hist[i])**2
        line1.set_ydata(prob_den)
        
        real_plt.set_ydata(psi_hist[i].real)
        imag_plt.set_ydata(psi_hist[i].imag)
        
        return line1, real_plt, imag_plt
    
    anim = FuncAnimation(fig, animate, frames=frames, 
                        interval=50, blit=True)
    
    plt.tight_layout()
    plt.close(fig)
    return anim

print("Creating animation...")
anim_tun = create_tun_anim(psi_hist, x, V, "Gaussian Wave Packet Tunneling", frame_step=10)
anim_tun.save('output1.mp4', writer='ffmpeg', fps=20, dpi=100)
HTML(anim_tun.to_jshtml())

In [ ]:
def T_th(E, V0, a, m=1, hbar=1):
    kappa = np.sqrt(2 * m * (V0 - E)) / hbar
    num = 1
    denom = 1 + (V0**2) / (4 * E * (V0 - E)) * np.sinh(kappa * a)**2
    return num / denom


def T_num(psi_hist, x, x_bar):
    prob_total = np.trapezoid(np.abs(psi_hist[0])**2, x)
    mask_right = x > x_bar + b_width/2
    prob_trans = np.trapezoid(np.abs(psi_hist[-1][mask_right])**2, x[mask_right])
    return prob_trans / prob_total


b_width_list = np.linspace(1, 15, 10)  
Tn_list = [] 
Tt_list = []  

E = 0.5 * k0**2  

print("Scanning different barrier widths for transmission...")

for a in b_width_list:
    V = V_pot(x, b_center, b_height, a)     
    H = build_H(N, dx, V, hbar, m)         
    U = time_evol_op(H, dt, hbar)           
    psi_hist = sim_wave(psi0, U, nt=800)    
    
   
    Tn = T_num(psi_hist, x, b_center)
    
    Tt = T_th(E, b_height, a)
    
    Tn_list.append(Tn)
    Tt_list.append(Tt)
    print(f"Barrier width = {a:.2f}, T_num = {Tn:.4f}, T_th = {Tt:.4f}")


plt.figure(figsize=(10,6))
plt.plot(b_width_list, Tt_list, 'b-o', label='Theoretical $T_{th}$')
plt.plot(b_width_list, Tn_list, 'r-s', label='Numerical $T_{num}$')
plt.xlabel('Barrier Width a')
plt.ylabel('Transmission Rate T')
plt.title('Comparison of Theoretical and Numerical Transmission')
plt.legend()
plt.grid(True)
plt.show()
